# 📚 WeebCentral Manga Downloader

Download manga chapters from WeebCentral with full parallel support.

### Output formats
| Option | Result |
|--------|--------|
| `pdf` | One PDF per chapter |
| `cbz` | CBZ archive with `ComicInfo.xml` (Kavita / Komga / CDisplayEx) |
| `images` | Raw image folder per chapter |
| `all` | PDF + CBZ + Images |

### Supported URL
```
https://weebcentral.com/series/SERIES_ID/manga-slug
```

In [4]:
!pip install -q requests 'httpx[http2]' nest_asyncio beautifulsoup4 lxml Pillow fpdf2 tqdm
!pip install PyPDF2
print('✅ Dependencies ready.')
!git clone https://github.com/Yui007/weebcentral_downloader
import sys
import os
from PyPDF2 import PdfMerger
from pathlib import Path

sys.path.insert(0, '/content/weebcentral_downloader/colab')

from colab_scraper    import scrape_manga_info, scrape_chapter_list
from colab_downloader import parse_chapter_selection, download_chapters

def merge_pdfs_to_target_size(pdf_files, output_path, target_size_mb=200, min_merge_threshold_mb=150):
    """
    Merge PDF files into larger files up to target_size_mb.
    Принудительно объединяет файлы меньше 150 МБ со следующими (не превышая 200 МБ).
    Returns list of created merged files.
    """
    target_bytes = target_size_mb * 1024 * 1024
    min_merge_bytes = min_merge_threshold_mb * 1024 * 1024
    merged_files = []

    if not pdf_files:
        return merged_files

    batch = []
    batch_size = 0
    batch_num = 1

    print(f"\n📊 Анализ файлов для объединения:")
    print(f"   Целевой размер: {target_size_mb} MB")
    print(f"   Порог объединения: {min_merge_threshold_mb} MB (файлы меньше этого будут объединяться)")
    print(f"   Найдено файлов: {len(pdf_files)}")

    i = 0
    while i < len(pdf_files):
        pdf_file = pdf_files[i]
        file_size = os.path.getsize(pdf_file)
        file_name = os.path.basename(pdf_file)

        # Если файл один больше лимита
        if file_size > target_bytes:
            print(f"   ⚠️  {file_name} ({file_size / (1024*1024):.1f} MB) - превышает лимит, будет сохранён отдельно")
            # Сохраняем текущий батч если есть
            if batch:
                merged_file = merge_pdf_batch(batch, output_path, batch_num)
                merged_files.append(merged_file)
                print(f"   ✅ Создан batch {batch_num}: {len(batch)} файлов, {batch_size / (1024*1024):.1f} MB")
                batch_num += 1
                batch = []
                batch_size = 0

            # Сохраняем большой файл отдельно
            merged_file = merge_pdf_batch([pdf_file], output_path, batch_num)
            merged_files.append(merged_file)
            print(f"   ✅ Создан batch {batch_num}: 1 файл, {file_size / (1024*1024):.1f} MB (отдельный файл)")
            batch_num += 1
            i += 1
            continue

        # Если текущий батч пустой, начинаем новый
        if not batch:
            batch = [pdf_file]
            batch_size = file_size
            i += 1
            continue

        # Пытаемся добавить текущий файл в существующий батч
        would_exceed = batch_size + file_size > target_bytes

        # Проверяем, нужно ли принудительно объединять (если текущий батч меньше порога)
        if batch_size < min_merge_bytes and not would_exceed:
            # Принудительно объединяем, даже если не превышает лимит
            batch.append(pdf_file)
            batch_size += file_size
            print(f"   🔗 Принудительное объединение: {os.path.basename(pdf_file)} ({(file_size / (1024*1024)):.1f} MB) -> batch {batch_num} (стало {batch_size / (1024*1024):.1f} MB)")
            i += 1
        elif would_exceed:
            # Если добавление превысит лимит, сохраняем текущий батч
            if batch:
                merged_file = merge_pdf_batch(batch, output_path, batch_num)
                merged_files.append(merged_file)
                print(f"   ✅ Создан batch {batch_num}: {len(batch)} файлов, {batch_size / (1024*1024):.1f} MB (достигнут лимит)")
                batch_num += 1

            # Начинаем новый батч
            batch = [pdf_file]
            batch_size = file_size
            i += 1
        else:
            # Обычное добавление
            batch.append(pdf_file)
            batch_size += file_size
            i += 1

    # Сохраняем последний батч
    if batch:
        merged_file = merge_pdf_batch(batch, output_path, batch_num)
        merged_files.append(merged_file)
        print(f"   ✅ Создан batch {batch_num}: {len(batch)} файлов, {batch_size / (1024*1024):.1f} MB (последний)")

    return merged_files

def merge_pdf_batch(pdf_list, output_dir, batch_num):
    """Merge a list of PDF files into one."""
    if not pdf_list:
        return None

    merger = PdfMerger()
    for pdf_file in pdf_list:
        merger.append(pdf_file)

    # Create output filename with batch number
    base_name = Path(pdf_list[0]).stem.split('_')[0]  # Get manga name from first file
    output_file = os.path.join(output_dir, f"{base_name}_batch_{batch_num}.pdf")

    # Ensure we don't overwrite existing files
    counter = 1
    while os.path.exists(output_file):
        output_file = os.path.join(output_dir, f"{base_name}_batch_{batch_num}_{counter}.pdf")
        counter += 1

    merger.write(output_file)
    merger.close()

    return output_file

def process_merged_pdfs(original_dir, merged_dir, target_size_mb=200):
    """Process all PDFs in original_dir and create merged versions in merged_dir."""
    # Create merged directory if it doesn't exist
    os.makedirs(merged_dir, exist_ok=True)

    # Find all PDF files in original directory
    pdf_files = sorted([os.path.join(original_dir, f) for f in os.listdir(original_dir)
                        if f.endswith('.pdf') and not f.startswith('merged_')])

    if not pdf_files:
        return []

    # Merge PDFs
    merged_files = merge_pdfs_to_target_size(pdf_files, merged_dir, target_size_mb)

    return merged_files

# ── 1. URL ───────────────────────────────────────────────────────────────────
SERIES_URL = input(
    '🌐 WeebCentral series URL\n'
    '   e.g. https://weebcentral.com/series/01JCZQE.../manga-slug\n'
    '   > '
).strip()

# ── 2. Scrape ────────────────────────────────────────────────────────────────
manga_info = scrape_manga_info(SERIES_URL)
chapters   = scrape_chapter_list(SERIES_URL)
total      = len(chapters)

# ── 3. Show chapter list ─────────────────────────────────────────────────────
print(f"\n{'='*62}")
print(f"  {'#':>4}   {'Chapter':<50}  Date")
print(f"{'='*62}")
for ch in chapters:
    num  = str(ch['index']).rjust(4)
    name = ch['title'][:50].ljust(50)
    print(f"  {num}   {name}  {ch['date']}")
print(f"{'='*62}")
print(f"  Total: {total} chapters\n")

# ── 4. Chapter selection ─────────────────────────────────────────────────────
print('📥 Selection formats:')
print('   all          → every chapter')
print('   single 5     → chapter 5 only')
print('   range 1-10   → chapters 1 through 10 (inclusive)')
print('   1,5,9,15     → specific chapters')
print()

while True:
    sel_input = input(f'🎯 Select chapters (1–{total}): ').strip()
    try:
        selected = parse_chapter_selection(sel_input, total)
        print(f'   ✅ {len(selected)} chapter(s) selected.')
        break
    except ValueError as e:
        print(f'   ❌ {e}\n   Try again.')

# ── 5. Output format ─────────────────────────────────────────────────────────
print()
print('📦 Output format:')
print('   pdf    → PDF file per chapter')
print('   cbz    → CBZ with ComicInfo.xml  (Kavita / Komga / CDisplayEx)')
print('   images → Raw image folder per chapter')
print('   all    → PDF + CBZ + Images')
print()

while True:
    fmt = input('🗂  Format [pdf / cbz / images / all]: ').strip().lower()
    if fmt in ('pdf', 'cbz', 'images', 'all'):
        break
    print('   ❌ Please enter pdf, cbz, images, or all.')

# ── 6. Download ──────────────────────────────────────────────────────────────
output_dir = download_chapters(
    manga_info       = manga_info,
    chapters         = chapters,
    selected_indices = selected,
    output_format    = fmt,
    output_dir       = '/content/weebcentral_downloader/colab/manga',
)

# ── 7. Merge PDFs if PDF format was requested ────────────────────────────────
if fmt in ('pdf', 'all'):
    print("\n" + "="*62)
    print("🔄 ОБЪЕДИНЕНИЕ PDF ФАЙЛОВ")
    print("="*62)
    print("📋 Правила объединения:")
    print("   • Максимальный размер файла: 200 MB")
    print("   • Файлы меньше 150 MB принудительно объединяются со следующими")
    print("   • Исходные файлы сохраняются в отдельной папке")
    print("="*62)

    # Define directories
    original_pdf_dir = output_dir  # PDFs are saved in the main output directory
    merged_pdf_dir = os.path.join(output_dir, 'merged_pdfs')

    # Process and merge PDFs
    merged_files = process_merged_pdfs(original_pdf_dir, merged_pdf_dir, target_size_mb=200)

    print("\n" + "="*62)
    print("📊 РЕЗУЛЬТАТЫ ОБЪЕДИНЕНИЯ")
    print("="*62)

    if merged_files:
        print(f"✅ Создано {len(merged_files)} объединённых PDF файлов в папке '{merged_pdf_dir}':")
        for i, mf in enumerate(merged_files, 1):
            size_mb = os.path.getsize(mf) / (1024 * 1024)
            print(f"   {i}. {os.path.basename(mf)} ({size_mb:.2f} MB)")

        print(f"\n📁 Исходные PDF файлы (по главам) сохранены в: {original_pdf_dir}")
        print(f"📁 Объединённые PDF файлы сохранены в: {merged_pdf_dir}")
        print("\n💡 Совет: При скачивании будут заархивированы только объединённые файлы")
    else:
        print("⚠️  PDF файлы для объединения не найдены.")

# ── 8. Download archive (only merged PDFs for PDF/ALL formats) ───────────────
print("\n" + "="*62)
print("📦 АРХИВАЦИЯ И СКАЧИВАНИЕ")
print("="*62)

try:
    # Import required modules for download
    import re
    import shutil
    from google.colab import files

    # Определяем директории
    original_pdf_dir = output_dir
    merged_pdf_dir = os.path.join(output_dir, 'merged_pdfs')

    # Проверяем, какой формат был выбран
    if fmt in ('pdf', 'all'):
        # Если есть склеенные файлы, архивируем только их
        if os.path.exists(merged_pdf_dir) and os.listdir(merged_pdf_dir):
            # Проверяем, есть ли в merged_pdf_dir PDF файлы
            merged_files = [f for f in os.listdir(merged_pdf_dir) if f.endswith('.pdf')]

            if merged_files:
                # Создаем безопасное имя для архива
                zip_stem = re.sub(r'[\\/:*?"<>|]', '_', manga_info['title']).strip() + '_merged_manga'
                zip_path = f'/content/weebcentral_downloader/colab/{zip_stem}'

                print(f'📦 Создание архива из объединённых PDF файлов: {merged_pdf_dir}')
                print(f'📁 Найдено {len(merged_files)} объединённых PDF файлов для архивации')

                # Создаем zip архив из папки merged_pdfs
                shutil.make_archive(zip_path, 'zip', merged_pdf_dir)

                # Полный путь к созданному архиву
                zip_file = f'{zip_path}.zip'

                if os.path.exists(zip_file):
                    # Получаем размер архива
                    size_mb = os.path.getsize(zip_file) / (1024 * 1024)
                    print(f'✅ Архив создан: {zip_stem}.zip ({size_mb:.2f} MB)')
                    print(f'📥 Начинаю скачивание...')

                    # Скачиваем архив
                    files.download(zip_file)

                    print('✅ Скачивание завершено!')
                    print(f'\n📌 Примечание: Архив содержит ТОЛЬКО объединённые PDF файлы (до 200MB каждый).')
                    print(f'   Исходные PDF файлы по главам сохранены в: {original_pdf_dir}')
                else:
                    print(f'❌ Файл архива не найден: {zip_file}')
            else:
                print(f'⚠️ В папке {merged_pdf_dir} не найдено PDF файлов')
        else:
            print(f'⚠️ Папка с объединёнными PDF файлами не найдена или пуста: {merged_pdf_dir}')
            print(f'📌 Нет объединённых PDF файлов для скачивания.')
    else:
        # Для других форматов (cbz, images) архивируем всё содержимое output_dir
        zip_stem = re.sub(r'[\\/:*?"<>|]', '_', manga_info['title']).strip() + '_manga'
        zip_path = f'/content/weebcentral_downloader/colab/{zip_stem}'

        print(f'📦 Создание архива из: {output_dir}')

        # Проверяем, существует ли директория
        if not os.path.exists(output_dir):
            print(f'❌ Ошибка: Директория {output_dir} не существует!')
        else:
            # Создаем zip архив
            shutil.make_archive(zip_path, 'zip', output_dir)

            # Полный путь к созданному архиву
            zip_file = f'{zip_path}.zip'

            if os.path.exists(zip_file):
                # Получаем размер архива
                size_mb = os.path.getsize(zip_file) / (1024 * 1024)
                print(f'✅ Архив создан: {zip_stem}.zip ({size_mb:.2f} MB)')
                print(f'📥 Начинаю скачивание...')

                # Скачиваем архив
                files.download(zip_file)

                print('✅ Скачивание завершено!')
            else:
                print(f'❌ Файл архива не найден: {zip_file}')

except Exception as e:
    print(f'❌ Ошибка при архивации/скачивании: {str(e)}')

print("\n" + "="*62)
print("✅ ВСЕ ОПЕРАЦИИ ЗАВЕРШЕНЫ")
print("="*62)

✅ Dependencies ready.
fatal: destination path 'weebcentral_downloader' already exists and is not an empty directory.
🌐 WeebCentral series URL
   e.g. https://weebcentral.com/series/01JCZQE.../manga-slug
   > https://weebcentral.com/series/01J76XYAQSGEJPXCSCVPQ3MHZM/One-Piece-Digital-Colored-Comics
🔎 Fetching series page: https://weebcentral.com/series/01J76XYAQSGEJPXCSCVPQ3MHZM/One-Piece-Digital-Colored-Comics

📖 Title    : One Piece (Color)
👤 Authors  : ODA Eiichiro
🏷  Tags     : Action, Adventure, Comedy, Drama, Fantasy ...
📌 Type     : Manga
📊 Status   : Ongoing
📅 Released : 1997
🖼  Cover    : https://temp.compsci88.com/cover/fallback/01J76XYAQSGEJPXCSCVPQ3MHZM.jpg

📝 Description:
As a child, Monkey D. Luffy dreamed of becoming the King of the Pirates. But his life changed when he accidentally gained the power to stretch like rubber...at the cost of never being able to swim again! Now Luffy, with the help of a motley collection of nakama, is setting off in search of "One Piec...

🔎 

  Ch053:   0%|          | 0/29 [00:00<?, ?pg/s]

       🖼  21 pages — downloading in parallel...


  Ch052:   0%|          | 0/21 [00:00<?, ?pg/s]

       🖼  20 pages — downloading in parallel...


...

  Ch100:   0%|          | 0/30 [00:00<?, ?pg/s]

          → One Piece (Color) - Ch098 - Chapter 98 Last Read 2024-09-07T17_04_15.717343Z.pdf

       ✅ 28/28 pages
       📄 Building PDF...
       ✅ 30/30 pages
       📄 Building PDF...
          → One Piece (Color) - Ch099 - Chapter 99 Last Read 2024-09-07T17_04_15.717343Z.pdf

          → One Piece (Color) - Ch100 - Chapter 100 Last Read 2024-09-07T17_04_15.717343Z.pdf

───────────────────────────────────────────────────────
🎉 Finished in 59s  —  saved to:
   /content/weebcentral_downloader/colab/manga/One Piece (Color)

🔄 ОБЪЕДИНЕНИЕ PDF ФАЙЛОВ
📋 Правила объединения:
   • Максимальный размер файла: 200 MB
   • Файлы меньше 150 MB принудительно объединяются со следующими
   • Исходные файлы сохраняются в отдельной папке

📊 Анализ файлов для объединения:
   Целевой размер: 200 MB
   Порог объединения: 150 MB (файлы меньше этого будут объединяться)
   Найдено файлов: 100
   🔗 Принудительное объединение: One Piece (Color) - Ch002 - Chapter 2 Last Read 2024-09-07T17_04_15.717343Z.pdf (14

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Скачивание завершено!

📌 Примечание: Архив содержит ТОЛЬКО объединённые PDF файлы (до 200MB каждый).
   Исходные PDF файлы по главам сохранены в: /content/weebcentral_downloader/colab/manga/One Piece (Color)

✅ ВСЕ ОПЕРАЦИИ ЗАВЕРШЕНЫ
